# SHIPP Stage 9B — AI Search v3 validation, sync, and semantic probe

**Purpose:** Validate the existing Delta Sync index, request a TRIGGERED sync when ready, and prove retrieval traces to Gold.

- Source: `bootcamp_students.shipp_gold.gold_listing_search_docs`
- Index: `bootcamp_students.shipp_gold.gold_listing_search_docs_index_v3`
- Endpoint: `shipp-vs`; embeddings from `databricks-gte-large-en`
- **This notebook does not create or delete endpoints or indexes.** Resource provisioning is a separately approved operation.
- Execute only after confirming that index v3 has been created. When first provisioning, this notebook can wait up to `INDEX_SYNC_TIMEOUT_SECONDS` for it to become ready.
- Output contract: notebook exits with a JSON `PASS` summary for `91_run_rag_pipeline`.

**Before deployment:** Review the exact Databricks workspace copy and verify `config/tables.py` is synced to v3. Running a local notebook file in VS Code is not the same as updating the Databricks workspace copy.

In [0]:
%run ../common/shipp_silver_lib

## 1. Load and validate runtime configuration

In [0]:
import importlib
import json
import time
from databricks.sdk import WorkspaceClient

from config import tables, settings

importlib.invalidate_caches()
importlib.reload(tables)
importlib.reload(settings)

# Reload retrieval after config.tables, otherwise its default index argument
# may retain the earlier v2 name in a long-running Python session.
for _name in [
    "rag.text", "rag.vision", "rag.content", "rag.search_docs",
    "rag.index", "rag.retrieval", "rag.semantic", "rag.evaluation",
]:
    importlib.reload(importlib.import_module(_name))

from rag import index as rag_index
from rag import retrieval as rag_retrieval
from rag.retrieval import search_listing_context

VS_INDEX_NAME = tables.VS_INDEX_NAME
GOLD_LISTING_SEARCH_DOCS = tables.GOLD_LISTING_SEARCH_DOCS
VS_ENDPOINT_NAME = settings.VS_ENDPOINT_NAME
EMBEDDING_ENDPOINT = settings.EMBEDDING_ENDPOINT
RAG_PROBE_QUERY = settings.RAG_PROBE_QUERY
RAG_PROBE_EXPECTED_LISTING = settings.RAG_PROBE_EXPECTED_LISTING

EXPECTED_INDEX = "bootcamp_students.shipp_gold.gold_listing_search_docs_index_v3"
EXPECTED_SOURCE = "bootcamp_students.shipp_gold.gold_listing_search_docs"
assert VS_INDEX_NAME == EXPECTED_INDEX, (
    f"STOP: runtime config points to {VS_INDEX_NAME!r}, expected {EXPECTED_INDEX!r}. "
    "Sync the approved config/tables.py in the Databricks workspace before running."
)
assert GOLD_LISTING_SEARCH_DOCS == EXPECTED_SOURCE, (
    f"STOP: unexpected Gold source: {GOLD_LISTING_SEARCH_DOCS!r}"
)
assert VS_ENDPOINT_NAME == "shipp-vs", f"STOP: unexpected endpoint: {VS_ENDPOINT_NAME!r}"
assert rag_retrieval.VS_INDEX_NAME == VS_INDEX_NAME, (
    "STOP: RAG retrieval imported a stale index name."
)

print("=== RUNTIME CODE ===")
print("tables.py:", tables.__file__)
print("settings.py:", settings.__file__)
print("rag/index.py:", rag_index.__file__)
print("rag/retrieval.py:", rag_retrieval.__file__)
print("INDEX:", VS_INDEX_NAME)
print("SOURCE:", GOLD_LISTING_SEARCH_DOCS)
print("CONFIG: PASS")

## 2. Validate existing endpoint, embedding model, and v3 index (read-only)

In [0]:
w = WorkspaceClient()

ep = w.vector_search_endpoints.get_endpoint(endpoint_name=VS_ENDPOINT_NAME)
endpoint_state = getattr(ep.endpoint_status.state, "value", ep.endpoint_status.state)
assert endpoint_state == "ONLINE", f"STOP: Vector Search endpoint is {endpoint_state}, not ONLINE"

embedding = w.serving_endpoints.get(EMBEDDING_ENDPOINT)
embedding_state = getattr(embedding.state.ready, "value", embedding.state.ready) if embedding.state else None
assert embedding_state == "READY", f"STOP: embedding endpoint is {embedding_state}, not READY"

idx = w.vector_search_indexes.get_index(index_name=VS_INDEX_NAME)
assert idx.endpoint_name == VS_ENDPOINT_NAME, (
    f"STOP: index attached to {idx.endpoint_name!r}, expected {VS_ENDPOINT_NAME!r}"
)
spec = idx.delta_sync_index_spec
assert spec is not None, "STOP: index is not a Delta Sync index"
assert spec.source_table == GOLD_LISTING_SEARCH_DOCS, (
    f"STOP: index source is {spec.source_table!r}, expected {GOLD_LISTING_SEARCH_DOCS!r}"
)

print("=== SHIPP V3 INDEX ===")
print("Endpoint:", ep.name, "ID:", ep.id, "status:", endpoint_state)
print("Embedding:", embedding.name, "status:", embedding_state)
print("Index:", idx.name)
print("Index ready:", idx.status.ready if idx.status else None)
print("Indexed rows:", idx.status.indexed_row_count if idx.status else None)
print("Message:", idx.status.message if idx.status else None)
print("RESOURCE CONTRACT: PASS")

## 3. Trigger synchronization if ready; wait for expected Gold count

In [0]:
expected_rows = spark.table(GOLD_LISTING_SEARCH_DOCS).count()
assert expected_rows > 0, "STOP: Gold source has no search documents"

status = rag_index.index_status(w, VS_INDEX_NAME)
assert status is not None, "STOP: index v3 is missing"

# A newly created index may be provisioning. Never trigger a new sync before it is ready.
# On subsequent executions, trigger sync even if the row count is unchanged: content can change.
sync_triggered = False
if status["ready"]:
    rag_index.trigger_sync(w, VS_INDEX_NAME)
    sync_triggered = True
    print("Triggered Delta Sync on existing READY index")
else:
    print("Index provisioning underway; waiting for initial synchronization")

# Counts/ready state are necessary gates, not a proof of end-to-end content freshness.
sync_seconds = rag_index.wait_until_synced(w, VS_INDEX_NAME, expected_rows)
final_status = rag_index.index_status(w, VS_INDEX_NAME)
assert final_status and final_status["ready"], "STOP: index did not become ready"
assert final_status["indexed_row_count"] == expected_rows, (
    f"STOP: index contains {final_status['indexed_row_count']} rows, Gold contains {expected_rows}"
)
print("INDEX SYNC STATUS: PASS")
print("Gold rows:", expected_rows)
print("Indexed rows:", final_status["indexed_row_count"])
print("Sync triggered:", sync_triggered)
print("Readiness/count wait seconds:", sync_seconds)

## 4. Semantic retrieval and Gold traceability

In [0]:
t0 = time.time()
results = search_listing_context(RAG_PROBE_QUERY, k=3, workspace_client=w)
probe_ms = int((time.time() - t0) * 1000)

print(f"Probe: {RAG_PROBE_QUERY!r} ({probe_ms} ms)")
for r in results:
    print(
        f"  score={(r['score'] or 0):.4f}  "
        f"listing_id={r['listing_id']}  title={r['title']}  "
        f"snippet={r['snippet'][:120]}"
    )

gold_ids = {
    r.listing_id
    for r in spark.table(GOLD_LISTING_SEARCH_DOCS).select("listing_id").collect()
}
assert results, "FAILED: semantic search returned no listings"
assert all(r["listing_id"] in gold_ids for r in results), (
    "FAILED: a search result does not trace to the Gold table"
)
assert RAG_PROBE_EXPECTED_LISTING in [r["listing_id"] for r in results], (
    f"FAILED: expected {RAG_PROBE_EXPECTED_LISTING} not in top 3. "
    "Investigate the evaluation seed/data and retrieval quality; do not simply remove the gate."
)

summary = {
    "index": VS_INDEX_NAME,
    "docs": expected_rows,
    "sync_seconds": sync_seconds,
    "probe_latency_ms": probe_ms,
    "status": "PASS",
}
print("SEMANTIC TRACEABILITY: PASS")
print("NOTE: sync_seconds is a readiness/count wait, not an end-to-end freshness measurement")
print(summary)
dbutils.notebook.exit(json.dumps(summary))